Test Bias and heat maps

In [2]:
!pip install gymnasium[mujoco] mujoco stable-baselines3
!pip -q install "gymnasium[mujoco]" stable-baselines3 tensorboard imageio matplotlib


In [3]:
# SINGLE-CELL COLAB: MuJoCo Ant PPO sweep (MLP sizes 16/32/64/128) + TensorBoard logs
# Logs: bias histograms, activation heatmaps, saturation maps over training time.
# Outputs: MP4 per hidden size showing heatmaps evolving over time.
#
# ✅ Run this cell, then in a new cell run:
#   %load_ext tensorboard
#   %tensorboard --logdir runs
#
# Notes:
# - Uses Stable-Baselines3 PPO for speed/robustness.
# - Captures activations from the policy MLP (tanh) using forward hooks.
# - Heatmaps are snapshots on a fixed observation batch taken from the env.


import os, time, math, shutil
import numpy as np
import torch
import torch.nn as nn
import gymnasium as gym
import matplotlib.pyplot as plt
import imageio.v2 as imageio

from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv, VecMonitor
from stable_baselines3.common.callbacks import BaseCallback
from stable_baselines3.common.logger import configure
from torch.utils.tensorboard import SummaryWriter

# ---------- Config ----------
ENV_ID = "Ant-v4"         # gymnasium mujoco env id
SEEDS = 0
#16, 32, 64, 128
HIDDEN_SIZES = [ 64]
TOTAL_TIMESTEPS = 200_000  # adjust (e.g., 500_000) if you want longer runs
LOG_EVERY_STEPS = 5_000    # snapshot cadence for TB + MP4 frames
EVAL_BATCH_N = 256         # fixed batch of observations for activation heatmaps
FPS = 8                    # mp4 fps
RUNS_DIR = "runs"
OUT_DIR = "out_videos"
os.makedirs(RUNS_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# ---------- Helpers ----------
def make_env(seed=0):
    def _thunk():
        env = gym.make(ENV_ID)
        env.reset(seed=seed)
        return env
    return _thunk

def collect_obs_batch(env, n=256):
    """Collect a fixed observation batch by rolling the env with random actions."""
    obs, _ = env.reset()
    obs_list = []
    for _ in range(n):
        a = env.action_space.sample()
        obs, _, term, trunc, _ = env.step(a)
        if term or trunc:
            obs, _ = env.reset()
        obs_list.append(obs)
    return np.asarray(obs_list, dtype=np.float32)

def fig_to_rgb_array(fig):
    fig.canvas.draw()
    img = np.asarray(fig.canvas.buffer_rgba())
    img = img[:, :, :3]   # drop alpha channel
    plt.close(fig)
    return img

def heatmap_image(mat, title, xlabel, ylabel):
    fig = plt.figure(figsize=(8, 3), dpi=140)
    ax = fig.add_subplot(111)
    ax.imshow(mat, aspect="auto")  # no explicit colors requested
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    fig.tight_layout()
    return fig_to_rgb_array(fig)

def line_image(xs, ys, title, xlabel, ylabel):
    fig = plt.figure(figsize=(6, 3), dpi=140)
    ax = fig.add_subplot(111)
    ax.plot(xs, ys)
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    fig.tight_layout()
    return fig_to_rgb_array(fig)

# ---------- Hook-based activation capture ----------
class ActivationProbe:
    """
    Attaches forward hooks to a list of nn.Modules and stores their outputs (activations).
    """
    def __init__(self):
        self.handles = []
        self.last_acts = {}  # name -> Tensor [B, H]
    def attach(self, module: nn.Module, name: str):
        def hook(mod, inp, out):
            # out can be Tensor or tuple
            t = out[0] if isinstance(out, (tuple, list)) else out
            if isinstance(t, torch.Tensor):
                self.last_acts[name] = t.detach()
        h = module.register_forward_hook(hook)
        self.handles.append(h)
    def clear(self):
        self.last_acts.clear()
    def close(self):
        for h in self.handles:
            h.remove()
        self.handles = []

def find_mlp_linears_and_tanh(policy: nn.Module):
    """
    SB3 ActorCriticPolicy has .mlp_extractor with .policy_net and .value_net (both Sequential).
    We'll hook Linear layers and tanh outputs in policy_net.
    """
    hooks = []
    # Prefer policy path (actor) for analysis:
    if hasattr(policy, "mlp_extractor") and hasattr(policy.mlp_extractor, "policy_net"):
        seq = policy.mlp_extractor.policy_net
    else:
        # fallback: hook all Linear layers in policy
        seq = policy
    # collect modules to hook
    named = []
    for i, m in enumerate(seq.modules()):
        if isinstance(m, nn.Linear):
            named.append((f"linear_{len(named)}", m))
        if isinstance(m, nn.Tanh):
            named.append((f"tanh_{len(named)}", m))
    return named

def get_bias_tensors(policy: nn.Module):
    biases = {}
    # biases from actor MLP (policy_net)
    if hasattr(policy, "mlp_extractor") and hasattr(policy.mlp_extractor, "policy_net"):
        for name, m in policy.mlp_extractor.policy_net.named_modules():
            if isinstance(m, nn.Linear) and m.bias is not None:
                biases[f"policy_net.{name}.bias"] = m.bias.detach().cpu()
    # action head bias (if present)
    if hasattr(policy, "action_net") and isinstance(policy.action_net, nn.Linear) and policy.action_net.bias is not None:
        biases["action_net.bias"] = policy.action_net.bias.detach().cpu()
    return biases


# ---------- Custom callback ----------
class TBProbeCallback(BaseCallback):
    def __init__(self, tb_writer: SummaryWriter, eval_obs_np: np.ndarray, probe: ActivationProbe,
                 run_tag: str, frames_list: list, log_every_steps: int):
        super().__init__()
        self.w = tb_writer
        self.eval_obs = torch.tensor(eval_obs_np, dtype=torch.float32)
        self.probe = probe
        self.run_tag = run_tag
        self.frames = frames_list
        self.log_every_steps = log_every_steps
        self.step_points = []
        self.sat_points = []  # global saturation over hooked tanh outputs

    def _on_step(self) -> bool:
        if (self.num_timesteps % self.log_every_steps) != 0:
            return True

        model = self.model
        policy = model.policy
        policy.eval()

        # Move eval obs to correct device
        obs_t = self.eval_obs.to(next(policy.parameters()).device)

        # Clear & run a forward pass through policy to populate activations
        self.probe.clear()
        with torch.no_grad():
            # SB3 policy forward expects obs in correct shape
            _ = policy(obs_t)

        # --- Bias logging ---
        biases = get_bias_tensors(policy)
        for bname, bt in biases.items():
            self.w.add_histogram(f"{self.run_tag}/bias/{bname}", bt.numpy(), global_step=self.num_timesteps)
            # Also log bias heatmap (layers concatenated handled below)
        # Combine biases into a single "bias heatmap" row-wise
        if biases:
            bias_rows = []
            bias_names = []
            for bname, bt in biases.items():
                v = bt.flatten().numpy()
                bias_rows.append(v)
                bias_names.append(bname)
            maxlen = max(len(v) for v in bias_rows)
            bias_mat = np.stack([np.pad(v, (0, maxlen - len(v))) for v in bias_rows], axis=0)
            img_bias = heatmap_image(
                bias_mat,
                title=f"{self.run_tag} | Bias heatmap @ {self.num_timesteps}",
                xlabel="bias index (padded)",
                ylabel="layer"
            )
            self.w.add_image(f"{self.run_tag}/heatmaps/bias", img_bias.transpose(2,0,1), self.num_timesteps)
        else:
            img_bias = None

        # --- Activation logging (heatmaps + saturation) ---
        # Build activation heatmaps for any hooked tensors with shape [B, H]
        act_mats = []
        act_labels = []
        sat_vals = []

        for name, t in sorted(self.probe.last_acts.items()):
            if t.ndim == 2:
                a = t.detach().cpu().numpy()
                # For tanh, saturation can be defined as |a| > 0.97
                if "tanh" in name:
                    sat = (np.abs(a) > 0.97).mean()
                    sat_vals.append(sat)
                # Normalize per-layer for visual stability (optional)
                # We'll just clip to percentiles for viewing:
                lo, hi = np.percentile(a, 1), np.percentile(a, 99)
                a_clip = np.clip(a, lo, hi)
                act_mats.append(a_clip)
                act_labels.append(name)

        # Create one combined activation heatmap by stacking layers vertically
        if act_mats:
            # Pad to same hidden width
            maxh = max(m.shape[1] for m in act_mats)
            stacked = np.concatenate([np.pad(m, ((0,0),(0,maxh-m.shape[1]))) for m in act_mats], axis=0)
            img_act = heatmap_image(
                stacked,
                title=f"{self.run_tag} | Activations (stacked) @ {self.num_timesteps}",
                xlabel="hidden unit",
                ylabel="(layers × batch)"
            )
            self.w.add_image(f"{self.run_tag}/heatmaps/activations", img_act.transpose(2,0,1), self.num_timesteps)

            # Saturation: mean across tanh outputs
            if sat_vals:
                sat_mean = float(np.mean(sat_vals))
            else:
                sat_mean = 0.0
            self.w.add_scalar(f"{self.run_tag}/saturation/tanh_frac(|a|>0.97)", sat_mean, self.num_timesteps)

            # Track for mp4 line plot
            self.step_points.append(self.num_timesteps)
            self.sat_points.append(sat_mean)

            # Compose a video frame: activations + bias + saturation curve
            # (1) activations heatmap, (2) bias heatmap (if any), (3) saturation curve
            imgs = [img_act]
            if img_bias is not None:
                imgs.append(img_bias)
            img_sat = line_image(self.step_points, self.sat_points,
                                 title=f"{self.run_tag} | tanh saturation over time",
                                 xlabel="timesteps", ylabel="frac(|a|>0.97)")
            imgs.append(img_sat)

            # stack images vertically
            widths = [im.shape[1] for im in imgs]
            W = max(widths)
            padded = []
            for im in imgs:
                h, w, c = im.shape
                if w < W:
                    pad = np.zeros((h, W-w, c), dtype=im.dtype)
                    im = np.concatenate([im, pad], axis=1)
                padded.append(im)
            frame = np.concatenate(padded, axis=0)
            self.frames.append(frame)

        policy.train()
        return True

# ---------- Main sweep ----------
for h in HIDDEN_SIZES:
    run_tag = f"ant_ppo_mlp{h}"
    logdir = os.path.join(RUNS_DIR, run_tag)
    if os.path.exists(logdir):
        shutil.rmtree(logdir, ignore_errors=True)
    os.makedirs(logdir, exist_ok=True)

    print(f"\n===== RUN {run_tag} =====")
    # env
    env = DummyVecEnv([make_env(SEEDS)])
    env = VecMonitor(env, filename=None)

    # fixed obs batch for heatmaps (use raw env once)
    raw_env = gym.make(ENV_ID)
    raw_env.reset(seed=SEEDS)
    eval_obs = collect_obs_batch(raw_env, n=EVAL_BATCH_N)
    raw_env.close()

    # SB3 logger + TB writer
    sb3_logger = configure(logdir, ["stdout", "tensorboard"])
    tbw = SummaryWriter(logdir)

    # Build PPO with tanh MLP policy
    policy_kwargs = dict(
        activation_fn=nn.Tanh,
        net_arch=dict(pi=[h, h], vf=[h, h]),
        ortho_init=False,
    )

    model = PPO(
        "MlpPolicy",
        env,
        policy_kwargs=policy_kwargs,
        verbose=1,
        tensorboard_log=logdir,
        n_steps=2048,
        batch_size=256,
        learning_rate=3e-4,
        gamma=0.99,
        gae_lambda=0.95,
        clip_range=0.2,
        device=device,
        seed=SEEDS,
    )
    model.set_logger(sb3_logger)

    # Attach activation hooks (actor mlp extractor)
    probe = ActivationProbe()
    for name, mod in find_mlp_linears_and_tanh(model.policy):
        probe.attach(mod, name=f"{name}")

    frames = []
    cb = TBProbeCallback(
        tb_writer=tbw,
        eval_obs_np=eval_obs,
        probe=probe,
        run_tag=run_tag,
        frames_list=frames,
        log_every_steps=LOG_EVERY_STEPS,
    )

    t0 = time.time()
    model.learn(total_timesteps=TOTAL_TIMESTEPS, callback=cb, progress_bar=True)
    dt = time.time() - t0
    print(f"[{run_tag}] done in {dt/60:.1f} min")

    # Save MP4
    mp4_path = os.path.join(OUT_DIR, f"{run_tag}_heatmaps.mp4")
    if len(frames) > 0:
        imageio.mimwrite(mp4_path, frames, fps=FPS)
        print("Wrote:", mp4_path)
    else:
        print("No frames recorded (increase TOTAL_TIMESTEPS or decrease LOG_EVERY_STEPS).")

    # Cleanup
    probe.close()
    tbw.flush()
    tbw.close()
    env.close()

print("\nAll runs complete.")
print("TensorBoard logs in:", RUNS_DIR)
print("MP4s in:", OUT_DIR)

Device: cpu

===== RUN ant_ppo_mlp64 =====
Logging to runs/ant_ppo_mlp64


Output()

Using cpu device
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 55.6     |
|    ep_rew_mean     | -61.5    |
| time/              |          |
|    fps             | 581      |
|    iterations      | 1        |
|    time_elapsed    | 3        |
|    total_timesteps | 2048     |
---------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 104         |
|    ep_rew_mean          | -119        |
| time/                   |             |
|    fps                  | 506         |
|    iterations           | 2           |
|    time_elapsed         | 8           |
|    total_timesteps      | 4096        |
| train/                  |             |
|    approx_kl            | 0.011210883 |
|    clip_fraction        | 0.15        |
|    clip_range           | 0.2         |
|    entropy_loss         | -11.3       |
|    explained_variance   | -0.00101    |
|    learning

[ant_ppo_mlp64] done in 7.7 min
Wrote: out_videos/ant_ppo_mlp64_heatmaps.mp4

All runs complete.
TensorBoard logs in: runs
MP4s in: out_videos


In [ ]:
# ============================================================
# FULL SINGLE CELL: Ant PPO MLP Sweep + 4 Metrics + Videos
# ============================================================

!pip install -q stable-baselines3[extra] gymnasium[mujoco] tensorboard imageio imageio-ffmpeg

import os, time, numpy as np, torch, gymnasium as gym, imageio, matplotlib.pyplot as plt
from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import BaseCallback
from stable_baselines3.common.vec_env import DummyVecEnv
from torch.utils.tensorboard import SummaryWriter
from IPython.display import clear_output

# ======================
# CONFIG
# ======================
#16, 32, 64, 128
HIDDEN_SIZES = [ 32]
TOTAL_TIMESTEPS = 200_000
LOG_INTERVAL = 5_000
EVAL_BATCH = 512
LOG_ROOT = "runs_ant_sweep"
VIDEO_DIR = "videos_ant"

os.makedirs(LOG_ROOT, exist_ok=True)
os.makedirs(VIDEO_DIR, exist_ok=True)

# ======================
# Activation Probe
# ======================

class ActivationProbe:
    def __init__(self, policy):
        self.last_acts = {}
        self.hooks = []
        idx = 0

        for m in policy.modules():
            if isinstance(m, (torch.nn.Linear, torch.nn.Tanh)):
                name = f"{type(m).__name__.lower()}_{idx}"
                idx += 1
                self.hooks.append(
                    m.register_forward_hook(self._hook(name))
                )

    def _hook(self, name):
        def fn(module, inp, out):
            if isinstance(out, torch.Tensor):
                self.last_acts[name] = out.detach()
        return fn

    def close(self):
        for h in self.hooks:
            h.remove()

# ======================
# Callback with ALL 4 metrics
# ======================

class ProbeCallback(BaseCallback):

    def __init__(self, writer, run_tag, eval_obs):
        super().__init__()
        self.w = writer
        self.run_tag = run_tag
        self.eval_obs = eval_obs
        self.probe = None

    def _on_training_start(self):
        self.probe = ActivationProbe(self.model.policy)

    def _on_step(self):

        if self.num_timesteps % LOG_INTERVAL != 0:
            return True

        policy = self.model.policy

        with torch.no_grad():
            obs = torch.tensor(self.eval_obs).float().to(policy.device)
            _ = policy(obs)

        act_mats = []
        act_labels = []

        for name, t in sorted(self.probe.last_acts.items()):
            if t.ndim == 2:
                A = t.detach().cpu().numpy()
                A = np.clip(A, -1, 1)
                act_mats.append(A)
                act_labels.append(name)

        # ---------------------------
        # Pick LAST TANH layer
        # ---------------------------
        A = None
        for nm in reversed(act_labels):
            if "tanh" in nm:
                A = act_mats[act_labels.index(nm)]
                break
        if A is None:
            A = act_mats[-1]

        A0 = A - A.mean(axis=0, keepdims=True)

        # ======================================================
        # 1. Useful neurons (variance)
        # ======================================================
        eps_var = 1e-4
        var = A0.var(axis=0)
        useful = (var > eps_var).sum()
        useful_frac = useful / A.shape[1]

        self.w.add_scalar(f"{self.run_tag}/useful_neurons", useful, self.num_timesteps)
        self.w.add_scalar(f"{self.run_tag}/useful_frac", useful_frac, self.num_timesteps)

        # ======================================================
        # 2. Saturation
        # ======================================================
        sat_thresh = 0.97
        sat = (np.abs(A) > sat_thresh)

        sat_neuron = sat.mean(axis=0)
        sat_neuron_count = (sat_neuron > 0.5).sum()
        sat_overall = sat.mean()

        self.w.add_scalar(f"{self.run_tag}/sat_overall", sat_overall, self.num_timesteps)
        self.w.add_scalar(f"{self.run_tag}/sat_neuron_count", sat_neuron_count, self.num_timesteps)

        # ======================================================
        # 3. Correlation / redundancy
        # ======================================================
        std = A0.std(axis=0) + 1e-8
        Z = A0 / std
        C = (Z.T @ Z) / max(1, Z.shape[0]-1)

        off = C - np.eye(C.shape[0])
        mean_corr = np.abs(off).mean()

        evals = np.linalg.eigvalsh(C)
        pr = (evals.sum()**2) / (np.square(evals).sum())

        self.w.add_scalar(f"{self.run_tag}/mean_abs_corr", mean_corr, self.num_timesteps)
        self.w.add_scalar(f"{self.run_tag}/participation_ratio", pr, self.num_timesteps)

        # ======================================================
        # 4. Effective Rank
        # ======================================================
        U,S,V = np.linalg.svd(A0.astype(np.float32), full_matrices=False)
        p = S / (S.sum()+1e-12)
        entropy = -(p*np.log(p+1e-12)).sum()
        eff_rank = np.exp(entropy)

        self.w.add_scalar(f"{self.run_tag}/effective_rank", eff_rank, self.num_timesteps)
        self.w.add_scalar(f"{self.run_tag}/effective_rank_frac", eff_rank/A.shape[1], self.num_timesteps)

        # ======================================================
        # Heatmap Video Frame
        # ======================================================
        fig = plt.figure(figsize=(6,4))
        plt.imshow(A.T, aspect="auto")
        plt.title(f"{self.run_tag} @ {self.num_timesteps}")
        plt.colorbar()
        fig.canvas.draw()

        img = np.frombuffer(fig.canvas.buffer_rgba(), dtype=np.uint8)
        img = img.reshape(fig.canvas.get_width_height()[::-1] + (4,))
        img = img[:,:,:3]
        plt.close(fig)

        video_path = f"{VIDEO_DIR}/{self.run_tag}.mp4"
        with imageio.get_writer(video_path, fps=4, mode='I') as writer:
            writer.append_data(img)

        return True

# ======================
# Train Loop
# ======================

env = DummyVecEnv([lambda: gym.make("Ant-v4")])

eval_obs = np.stack([env.reset()[0] for _ in range(EVAL_BATCH)])

for H in HIDDEN_SIZES:

    run_tag = f"ppo_mlp{H}"
    writer = SummaryWriter(f"{LOG_ROOT}/{run_tag}")

    model = PPO(
        "MlpPolicy",
        env,
        policy_kwargs=dict(net_arch=[H,H]),
        verbose=0
    )

    cb = ProbeCallback(writer, run_tag, eval_obs)

    print("Training", run_tag)
    model.learn(TOTAL_TIMESTEPS, callback=cb)

    writer.close()

# ======================
# Launch TensorBoard
# ======================

%load_ext tensorboard
%tensorboard --logdir runs_ant_sweep

In [4]:
# ============================================================
# Install dependencies
# ============================================================

!pip install -q stable-baselines3[extra] gymnasium[mujoco] imageio imageio-ffmpeg

# ============================================================
# Imports
# ============================================================

import os
import numpy as np
import torch
import gymnasium as gym
import imageio
import matplotlib.pyplot as plt

from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import BaseCallback
from stable_baselines3.common.vec_env import DummyVecEnv

# ============================================================
# Config
# ============================================================

HIDDEN = 32
TOTAL_TIMESTEPS = 200_000
LOG_INTERVAL = 5_000
EVAL_BATCH = 512
VIDEO_FILE = "ant_metrics_overlay.mp4"

# ============================================================
# Activation Probe
# ============================================================

class ActivationProbe:
    def __init__(self, policy):
        self.last = {}
        self.hooks = []
        idx = 0
        for m in policy.modules():
            if isinstance(m, (torch.nn.Linear, torch.nn.Tanh)):
                name = f"{type(m).__name__.lower()}_{idx}"
                idx += 1
                self.hooks.append(m.register_forward_hook(self._hook(name)))

    def _hook(self, name):
        def fn(module, inp, out):
            if isinstance(out, torch.Tensor):
                self.last[name] = out.detach()
        return fn

# ============================================================
# Callback With Video + 4 Metrics Overlay
# ============================================================

class VideoMetricsCallback(BaseCallback):

    def __init__(self, eval_obs):
        super().__init__()
        self.eval_obs = eval_obs
        self.probe = None
        self.frames = []

    def _on_training_start(self):
        self.probe = ActivationProbe(self.model.policy)

    def _on_step(self):

        if self.num_timesteps % LOG_INTERVAL != 0:
            return True

        policy = self.model.policy

        with torch.no_grad():
            obs = torch.tensor(self.eval_obs).float().to(policy.device)
            _ = policy(obs)

        # pick last tanh
        A = None
        for k in reversed(sorted(self.probe.last.keys())):
            if "tanh" in k:
                A = self.probe.last[k].cpu().numpy()
                break
        if A is None:
            return True

        A0 = A - A.mean(axis=0, keepdims=True)

        # ====================================================
        # 1. Useful neurons
        # ====================================================
        var = A0.var(axis=0)
        useful = int((var > 1e-4).sum())
        useful_frac = useful / A.shape[1]

        # ====================================================
        # 2. Saturation
        # ====================================================
        sat = (np.abs(A) > 0.97)
        sat_overall = sat.mean()
        sat_neurons = int((sat.mean(axis=0) > 0.5).sum())

        # ====================================================
        # 3. Correlation
        # ====================================================
        std = A0.std(axis=0) + 1e-8
        Z = A0 / std
        C = (Z.T @ Z) / max(1, Z.shape[0]-1)
        off = C - np.eye(C.shape[0])
        mean_corr = np.abs(off).mean()

        evals = np.linalg.eigvalsh(C)
        pr = (evals.sum()**2) / (np.square(evals).sum())

        # ====================================================
        # 4. Effective Rank
        # ====================================================
        U,S,V = np.linalg.svd(A0.astype(np.float32), full_matrices=False)
        p = S / (S.sum()+1e-12)
        entropy = -(p*np.log(p+1e-12)).sum()
        eff_rank = np.exp(entropy)

        # ====================================================
        # Create Frame
        # ====================================================

        fig = plt.figure(figsize=(8,5))
        ax = fig.add_subplot(111)

        ax.imshow(A.T, aspect="auto")
        ax.set_title(f"Ant PPO MLP-{HIDDEN} @ {self.num_timesteps}")
        ax.set_xlabel("Batch")
        ax.set_ylabel("Neuron")

        text = (
            f"Useful: {useful}/{A.shape[1]} ({useful_frac:.2f})\n"
            f"Sat overall: {sat_overall:.3f}\n"
            f"Sat neurons: {sat_neurons}\n"
            f"Mean |corr|: {mean_corr:.3f}\n"
            f"Participation ratio: {pr:.2f}\n"
            f"Effective rank: {eff_rank:.2f}"
        )

        ax.text(
            1.02, 0.5, text,
            transform=ax.transAxes,
            fontsize=9,
            verticalalignment='center'
        )

        fig.tight_layout()
        fig.canvas.draw()

        img = np.frombuffer(fig.canvas.buffer_rgba(), dtype=np.uint8)
        img = img.reshape(fig.canvas.get_width_height()[::-1] + (4,))
        img = img[:,:,:3]
        plt.close(fig)

        self.frames.append(img)

        return True

    def save_video(self):
        imageio.mimwrite(VIDEO_FILE, self.frames, fps=4)

# ============================================================
# Setup Environment
# ============================================================

env = DummyVecEnv([lambda: gym.make("Ant-v4")])
eval_obs = np.stack([env.reset()[0] for _ in range(EVAL_BATCH)])

# ============================================================
# Train
# ============================================================

model = PPO(
    "MlpPolicy",
    env,
    policy_kwargs=dict(net_arch=[HIDDEN, HIDDEN]),
    verbose=1
)

callback = VideoMetricsCallback(eval_obs)

model.learn(TOTAL_TIMESTEPS, callback=callback)

callback.save_video()

print("Saved video:", VIDEO_FILE)

Using cpu device
-----------------------------
| time/              |      |
|    fps             | 685  |
|    iterations      | 1    |
|    time_elapsed    | 2    |
|    total_timesteps | 2048 |
-----------------------------
-----------------------------------------
| time/                   |             |
|    fps                  | 493         |
|    iterations           | 2           |
|    time_elapsed         | 8           |
|    total_timesteps      | 4096        |
| train/                  |             |
|    approx_kl            | 0.010276161 |
|    clip_fraction        | 0.0742      |
|    clip_range           | 0.2         |
|    entropy_loss         | -11.3       |
|    explained_variance   | -0.0301     |
|    learning_rate        | 0.0003      |
|    loss                 | 121         |
|    n_updates            | 10          |
|    policy_gradient_loss | -0.0223     |
|    std                  | 0.983       |
|    value_loss           | 265         |
-----------------

Saved video: ant_metrics_overlay.mp4
